In [65]:
import pandas as pd

In [66]:
regular_season = pd.read_csv("kaggle_data/MRegularSeasonCompactResults.csv")
regular_season = regular_season[regular_season['Season']>=2003]

In [67]:
wins = regular_season[['Season','DayNum','WTeamID','LTeamID','WScore',"LScore"]].copy()
wins.columns = ["Season", "DayNum", "TeamID", "OpponentID","PointsScored","PointsAllowed"]
wins["Win"] = 1
wins["Loss"] = 0

In [68]:
losses = regular_season[['Season','DayNum','LTeamID','WTeamID','LScore','WScore']].copy()
losses.columns = ['Season','DayNum','TeamID','OpponentID',"PointsScored",'PointsAllowed']
losses["Win"] = 0
losses['Loss'] = 1

In [69]:
games =  pd.concat([wins,losses],ignore_index=True)
games['PointDiff'] = games['PointsScored'] - games['PointsAllowed']

In [70]:
team_season = (
    games.groupby(['Season','TeamID']).agg(
        Wins=('Win','sum'),
        Losses = ('Loss',"sum"),
        OpponentIDs = ('OpponentID',list)
    ).reset_index()
)

In [71]:
team_season['winPer']= team_season['Wins']/(team_season['Wins']+team_season['Losses'])
team_season

,Season,TeamID,Wins,Losses,OpponentIDs,winPer
0,2003,1102,12,16,"[1391, 1117, 1410, 1204, 1125, 1431, 1204, 129...",0.428571
1,2003,1103,13,14,"[1324, 1233, 1347, 1214, 1123, 1138, 1296, 118...",0.481481
2,2003,1104,17,11,"[1328, 1106, 1292, 1326, 1422, 1382, 1132, 134...",0.607143
3,2003,1105,7,19,"[1366, 1289, 1212, 1115, 1115, 1289, 1411, 143...",0.269231
4,2003,1106,13,15,"[1209, 1128, 1238, 1212, 1115, 1290, 1105, 138...",0.464286
...,...,...,...,...,...,...
8341,2026,1477,9,21,"[1192, 1119, 1146, 1368, 1410, 1230, 1322, 132...",0.300000
8342,2026,1478,14,17,"[1310, 1248, 1284, 1127, 1384, 1481, 1152, 125...",0.451613
8343,2026,1479,15,17,"[1260, 1288, 1127, 1478, 1447, 1148, 1481, 115...",0.468750
8344,2026,1480,13,17,"[1154, 1399, 1367, 1407, 1468, 1184, 1316, 147...",0.433333


In [72]:
team_season['SOS'] = 0.0

lookup_dict = team_season.set_index(
    ['Season','TeamID']
).to_dict('index')

for idx, team in team_season.iterrows():
    sos = 0
    season = team['Season']

    for opp_id in team['OpponentIDs']:
        opp = lookup_dict[(season,opp_id)]

        sos += (2/3)*opp['winPer']

        for oppopp_id in opp['OpponentIDs']:
            oppopp = lookup_dict[(season,oppopp_id)]
            sos += (1/3)*oppopp['winPer']

    team_season.loc[idx,'SOS'] = sos
    

In [73]:
season_sos = team_season.groupby('Season')['SOS'].mean()


team_season['SOS']=(
    team_season['SOS'] - team_season['Season'].map(season_sos)
    
).round(5)
team_season

,Season,TeamID,Wins,Losses,OpponentIDs,winPer,SOS
0,2003,1102,12,16,"[1391, 1117, 1410, 1204, 1125, 1431, 1204, 129...",0.428571,4.02880
1,2003,1103,13,14,"[1324, 1233, 1347, 1214, 1123, 1138, 1296, 118...",0.481481,-6.64408
2,2003,1104,17,11,"[1328, 1106, 1292, 1326, 1422, 1382, 1132, 134...",0.607143,19.07337
3,2003,1105,7,19,"[1366, 1289, 1212, 1115, 1115, 1289, 1411, 143...",0.269231,-25.18929
4,2003,1106,13,15,"[1209, 1128, 1238, 1212, 1115, 1290, 1105, 138...",0.464286,-14.90442
...,...,...,...,...,...,...,...
8341,2026,1477,9,21,"[1192, 1119, 1146, 1368, 1410, 1230, 1322, 132...",0.300000,-17.52578
8342,2026,1478,14,17,"[1310, 1248, 1284, 1127, 1384, 1481, 1152, 125...",0.451613,-18.68163
8343,2026,1479,15,17,"[1260, 1288, 1127, 1478, 1447, 1148, 1481, 115...",0.468750,-15.99969
8344,2026,1480,13,17,"[1154, 1399, 1367, 1407, 1468, 1184, 1316, 147...",0.433333,-14.04961


In [ ]:
point_diff = (
    games.groupby(['Season','TeamID']).agg(
        TotalPointDiff=('PointDiff',sum),
        AvgPointDiff=('PointDiff','mean')
    ).reset_index()
)

team_season = team_season.merge(point_diff, on = ['Season','TeamID'],how='left')

team_season

team_season['SRS'] = team_season['AvgPointDiff'] + team_season['SOS']

In [76]:
team_season.to_csv('season_stats.csv',index=False) 